In [8]:
# Simple criteria to distinguish SC neurons and RGCs:
# Classify neurons as RGC that meet at least 2 criteria:
# 1. spatial_extent_10pct > 300 µm 
# 2. trough_to_peak_time < 0.24 ms
# 3. n_fast_peaks > 1

# How to label positive waveforms?

import numpy as np
import pandas as pd
import os
from scipy.io import loadmat, savemat
from scipy.signal import find_peaks

import matplotlib.pyplot as plt
from matplotlib.backends.backend_pdf import PdfPages

from tqdm.notebook import tqdm

sr = 30000
ns_1ms = int(1.0 * sr / 1000)

mat = loadmat('/Users/iii9781/viewer/scripts/Mouse10_20260210_795to2220.mat')['wf_feat']
mat

array([[(array([[  0,   1,   2,   3,   4,   5,   6,   7,   8,   9,  10,  11,  12,
                 13,  14,  15,  16,  17,  18,  19,  20,  21,  22,  23,  24,  25,
                 26,  27,  28,  29,  30,  31,  32,  33,  34,  35,  36,  37,  38,
                 39,  40,  41,  42,  43,  44,  45,  46,  47,  48,  49,  50,  51,
                 52,  53,  54,  55,  56,  57,  58,  59,  60,  61,  62,  63,  64,
                 65,  66,  67,  68,  69,  70,  71,  72,  73,  74,  75,  76,  77,
                 78,  79,  80,  81,  82,  83,  84,  85,  86,  87,  88,  89,  90,
                 91,  92,  93,  94,  95,  96,  97,  98,  99, 100, 101, 102, 103,
                104, 105, 106, 107, 108, 109, 110, 111, 112, 113, 114, 115, 116,
                117, 118, 119, 120, 121, 122, 123, 124, 125, 126, 127, 128, 129,
                130, 131, 132, 133, 134, 135, 136, 137, 138, 139, 140, 141, 142,
                143, 144, 145, 146, 147, 148, 149, 150, 151, 152, 153, 154, 155,
                156, 157, 1

In [2]:
d = d['meanWaveforms']
mw = d[0, 0]

# Loop over all fields
for field in mw.dtype.names:
    print(f"{field}: {mw[field].shape}")

meanWav = mw['data']

d.dtype.names
print("\nmeanWav: ",meanWav.shape)

unitIds = mw['unitIds'].ravel()
chanmap = mw['chanmap'].flatten()
xpos = mw['xpos'].flatten()
ypos = mw['ypos'].flatten()
ptp_chan_idx = mw['ptp_chan_idx'].flatten()
timepts = mw['timepts'].flatten()
sbefore = mw['sbefore'].item()
safter = mw['safter'].item()

nCh, nS, nUnits = meanWav.shape
num_channels_to_plot = 21  # number of channels around peak to include
spatialWindow = 300 # micrometers

data: (384, 120, 162)
samplenum: (1, 1)
sbefore: (1, 1)
safter: (1, 1)
unitIds: (162, 1)
chanmap: (1, 384)
min_chan_idx: (1, 162)
ptp_chan_idx: (1, 162)
timepts: (1, 120)
xpos: (1, 384)
ypos: (1, 384)

meanWav:  (384, 120, 162)


In [3]:
# function to get waveforms across all channels, not just the 21 channels

def get_multichannel_waveform_wf_feat_uncapped(unit_idx, meanWav, xpos, ypos, 
                                                amplitude_threshold=0.1):
    """
    Use amplitude threshold instead of fixed channel count.
    """
    mean_wave = meanWav[:, :, unit_idx]
    
    # Find peak channel (max absolute amplitude)
    # peak_ch = np.argmax(np.max(np.abs(mean_wave), axis=1))
    unit_id = unitIds[unit_idx]
    peak_ch_i = ptp_chan_idx[unit_idx]-1
    peak_waveform = mean_wave[peak_ch_i, :]
    peak_amp = np.max(np.abs(peak_waveform))
    
    peak_x = xpos[peak_ch_i]
    peak_y = ypos[peak_ch_i]
    
    # Get same-shank channels
    uxp = np.unique(xpos).astype(int)
    this_shank_xpos = uxp[np.abs(uxp - peak_x) < 200]
    sameshank = np.where(np.isin(xpos, this_shank_xpos))[0]
    
    # Instead of fixed num_channels, use ALL channels above amplitude threshold
    max_amps = np.max(np.abs(mean_wave[sameshank, :]), axis=1)
    significant_mask = max_amps > (amplitude_threshold * peak_amp)
    channels_to_use = sameshank[significant_mask]
    
    # Get multi-channel waveforms sorted by depth (deep to superficial)
    depth_order = np.argsort(ypos[channels_to_use])
    sorted_channels = channels_to_use[depth_order]
    multichan_waveforms = mean_wave[sorted_channels, :]

    return {
        'unit_id': unit_id,
        'peak_waveform': peak_waveform,
        'peak_ch': peak_ch_i,
        'ch_idxs': sorted_channels,
        'channels': chanmap[sorted_channels],
        'multichan_waveforms': multichan_waveforms,
        'channel_depths': ypos[sorted_channels],
        'peak_depth': peak_y,
        'peak_amp': peak_amp
    }

In [4]:
# Extract multi-channel waveform wf_feat for all units
unit_data = []
for i in tqdm(range(nUnits), desc="Extracting multi-channel waveforms"):
    data = get_multichannel_waveform_wf_feat_uncapped(i, meanWav, xpos, ypos)
    unit_data.append(data)

Extracting multi-channel waveforms:   0%|          | 0/162 [00:00<?, ?it/s]

In [5]:
# ==============================================================================
# IMPROVED SPATIAL SPREAD - Multiple Thresholds
# ==============================================================================

def calculate_spatial_spread_improved(multichan_waveforms, channel_depths, peak_depth, peak_amp):
    """
    Calculate spatial spread with multiple amplitude thresholds.
    """
    wf_feat = {}
    
    max_amps_per_channel = np.max(np.abs(multichan_waveforms), axis=1)
    
    # Multiple thresholds to capture spread at different levels
    thresholds = {
        '10pct': 0.1,
        '20pct': 0.2,
        '30pct': 0.3,
        '50pct': 0.5
    }
    
    for name, thresh in thresholds.items():
        active = max_amps_per_channel > (thresh * peak_amp)
        wf_feat[f'n_active_channels_{name}'] = np.sum(active)
        
        if np.sum(active) > 1:
            active_depths = channel_depths[active]
            wf_feat[f'spatial_extent_{name}'] = np.max(active_depths) - np.min(active_depths)
        else:
            wf_feat[f'spatial_extent_{name}'] = 0
    
    # Weighted spatial spread (amplitude-weighted standard deviation)
    if len(max_amps_per_channel) > 1:
        weights = max_amps_per_channel / np.sum(max_amps_per_channel)
        weighted_center = np.sum(weights * channel_depths)
        weighted_variance = np.sum(weights * (channel_depths - weighted_center)**2)
        wf_feat['spatial_spread_weighted_std'] = np.sqrt(weighted_variance)
    else:
        wf_feat['spatial_spread_weighted_std'] = 0
    
    # Amplitude decay rate (simple linear regression in log space)
    peak_ch_idx = np.argmax(max_amps_per_channel)
    distances = np.abs(channel_depths - channel_depths[peak_ch_idx])
    
    # Only use channels with measurable amplitude
    valid_mask = max_amps_per_channel > (0.05 * peak_amp)
    if np.sum(valid_mask) > 3:
        valid_distances = distances[valid_mask]
        valid_amps = max_amps_per_channel[valid_mask]
        
        # Log-linear fit: log(amp) ~ -distance/decay_constant
        log_amps = np.log(valid_amps + 1e-10)
        from scipy.stats import linregress
        slope, intercept, r_value, _, _ = linregress(valid_distances, log_amps)
        
        # Decay length is -1/slope (more negative slope = faster decay = smaller spread)
        if slope < 0:
            wf_feat['decay_length'] = -1 / slope
        else:
            wf_feat['decay_length'] = 1000  # Very slow decay
        wf_feat['decay_r_squared'] = r_value**2
    else:
        wf_feat['decay_length'] = 0
        wf_feat['decay_r_squared'] = 0
    
    return wf_feat


# Add this function near your other feature functions
def calculate_conduction_velocity(multichan_waveforms, channel_depths, sr=30000):
    n_channels = multichan_waveforms.shape[0]
    center = n_channels // 2
    # Use a small window (3 up/3 down) to find the slope
    idx = np.arange(max(0, center-3), min(n_channels, center+4))
    
    trough_samples = np.argmin(multichan_waveforms[idx, :], axis=1)
    trough_ms = trough_samples / sr * 1000
    depths = channel_depths[idx]
    
    from scipy.stats import linregress
    slope, _, r_val, _, _ = linregress(depths, trough_ms)
    return abs(slope), r_val**2


In [6]:
## aligned cluster average (calculates tilt, tilt is 0 for SC, has a slope for RGC)
def get_aligned_cluster_average(cluster_unit_indices, unit_data, window_size=41):
    """
    Aligns units by their peak channel and averages them.
    
    window_size: Total number of channels in the output (should be odd).
    """
    half_win = window_size // 2
    aligned_waveforms = []
    
    for idx in cluster_unit_indices:
        unit = unit_data[idx]
        wf = unit['multichan_waveforms']  # shape: (n_channels, n_samples)
        
        # Identify where the peak channel is relative to the extracted channels
        # In your 'uncapped' function, the peak_ch is usually centered, 
        # but let's be explicit:
        max_amps = np.max(np.abs(wf), axis=1)
        peak_idx_in_wf = np.argmax(max_amps)
        
        n_ch, n_samples = wf.shape
        
        # Create an empty canvas (padded with zeros)
        canvas = np.zeros((window_size, n_samples))
        
        # Calculate bounds to map 'wf' into 'canvas' centered on peak_idx_in_wf
        # We want peak_idx_in_wf to land at half_win in the canvas
        start_canvas = max(0, half_win - peak_idx_in_wf)
        end_canvas = min(window_size, half_win + (n_ch - peak_idx_in_wf))
        
        start_wf = max(0, peak_idx_in_wf - half_win)
        end_wf = min(n_ch, peak_idx_in_wf + (window_size - half_win))
        
        # Ensure the slice sizes match
        slice_len = min(end_canvas - start_canvas, end_wf - start_wf)
        canvas[start_canvas:start_canvas+slice_len, :] = wf[start_wf:start_wf+slice_len, :]
        
        aligned_waveforms.append(canvas)
    
    if not aligned_waveforms:
        return np.zeros((window_size, 90)) # fallback
        
    return np.mean(aligned_waveforms, axis=0)

In [7]:
# EXTRACT TEMPORAL wf_feat (Fast + Slow Components)

def extract_temporal_wf_feat(waveform, sr, sbefore):
    """
    Extract temporal wf_feat distinguishing RGC fast bi/triphasic + slow trough
    from SC biphasic waveforms.
    """
    wf_feat = {}
    
    # Time vector in ms
    t_ms = (np.arange(len(waveform)) - sbefore) / sr * 1000
    
    # === PRIMARY TROUGH (main negative deflection) ===
  
    primary_trough_idx = np.argmin(waveform[sbefore:sbefore+(ns_1ms//3)]) + sbefore
    primary_trough_amp = waveform[primary_trough_idx]
    primary_trough_time = t_ms[primary_trough_idx]
    
    wf_feat['primary_trough_idx'] = primary_trough_idx
    wf_feat['primary_trough_amp'] = abs(primary_trough_amp)
    wf_feat['primary_trough_time'] = primary_trough_time
    
    # === DETECT SECONDARY SLOW TROUGH (characteristic of RGCs) ===
    # Look for a second trough after the primary peak
    primary_peak_idx = np.argmax(waveform[primary_trough_idx:sbefore+ns_1ms]) + primary_trough_idx # primary_peak: Repolarisation peak
    primary_peak_amp = waveform[primary_peak_idx]
    
    # Search for secondary trough after primary peak (in the later 1-2 ms)
    search_start = primary_peak_idx + int(0.3 * sr / 1000)  # start 0.3ms after peak
    search_end = min(len(waveform), primary_peak_idx + int(2.5 * sr / 1000))  # search up to 2.5ms
    
    if search_end > search_start:
        late_segment = waveform[search_start:search_end]
        secondary_trough_idx_rel = np.argmin(late_segment)
        secondary_trough_idx = search_start + secondary_trough_idx_rel
        secondary_trough_amp = waveform[secondary_trough_idx]
        
        # Only count as secondary trough if significantly negative
        if secondary_trough_amp < -0.15 * abs(primary_trough_amp):
            wf_feat['has_secondary_trough'] = 1
            wf_feat['secondary_trough_amp'] = abs(secondary_trough_amp)
            wf_feat['secondary_trough_time'] = t_ms[secondary_trough_idx]
            wf_feat['secondary_to_primary_ratio'] = abs(secondary_trough_amp) / abs(primary_trough_amp)
        else:
            wf_feat['has_secondary_trough'] = 0
            wf_feat['secondary_trough_amp'] = 0
            wf_feat['secondary_trough_time'] = np.nan
            wf_feat['secondary_to_primary_ratio'] = 0
    else:
        wf_feat['has_secondary_trough'] = 0
        wf_feat['secondary_trough_amp'] = 0
        wf_feat['secondary_trough_time'] = np.nan
        wf_feat['secondary_to_primary_ratio'] = 0
    
    # === FAST COMPONENT DETECTION (bi/triphasic) ===
    # Look at the first 1ms after spike onset for fast oscillations
    fast_window = waveform[ns_1ms-1:ns_1ms*3+5] 

    # Count zero crossings (indicator of bi/triphasic nature)
    zero_crossings = np.sum(np.diff(np.sign(fast_window)) != 0)
    wf_feat['fast_zero_crossings'] = zero_crossings

    max_peak_idx = np.argmax(waveform) # max_peak: max peak in the whole waveform
    max_peak_amp = waveform[max_peak_idx]
    
    # Detect peaks in fast component
    peaks, _ = find_peaks(fast_window, distance=int(0.2*ns_1ms), height=max_peak_amp/4)
    amps = fast_window[peaks]
    
    # If more than 2 peaks, keep primary_peak and preceding peak
    if len(peaks) > 2:      
        if primary_peak_amp in amps:
            idx = np.where(amps == primary_peak_amp)[0][0]
            if idx > 0: 
                idx = np.array([idx-1,idx])
                peaks = peaks[idx]
                amps = amps[idx]
            else: 
                peaks = peaks[[0]]
                amps = amps[[0]]
        else:
            idx = np.argmax(amps) 
            peaks = peaks[[idx]]
            amps = amps[[idx]]

    # Remove 'peak' that occurs after repolarisation
    if len(peaks) > 1:
        if peaks[0] == primary_peak_idx-(ns_1ms-1):
            peaks = peaks[[0]] 
            amps = amps[[0]]
    
    wf_feat['n_fast_peaks'] = len(peaks)
    wf_feat['peaks'] = peaks + ns_1ms-1
    wf_feat['peaks_amp'] = amps
    
    # === DURATION wf_feat ===
    # Trough-to-peak time
    wf_feat['trough_to_peak_time'] = (primary_peak_idx - primary_trough_idx) / sr * 1000
    
    # Half-width
    half_amp = primary_trough_amp / 2
    crossings = np.where(np.diff(np.sign(waveform - half_amp)))[0]
    if len(crossings) >= 2:
        wf_feat['half_width'] = (crossings[-1] - crossings[0]) / sr * 1000
    else:
        wf_feat['half_width'] = np.nan
    
    # Total duration (10-90% recovery)
    recovery_90 = 0.1 * primary_trough_amp
    post_trough = waveform[primary_trough_idx:]
    recovery_idx = np.where(post_trough >= recovery_90)[0]
    if len(recovery_idx) > 0:
        wf_feat['duration_10_90'] = recovery_idx[0] / sr * 1000
    else:
        wf_feat['duration_10_90'] = np.nan
    
    # === REPOLARIZATION wf_feat ===
    if primary_peak_idx + 10 < len(waveform):
        repol_slope = (waveform[primary_peak_idx + 10] - waveform[primary_peak_idx]) / (10 / sr * 1000)
        wf_feat['repol_slope'] = repol_slope
    else:
        wf_feat['repol_slope'] = np.nan
    
    # Peak-to-trough ratio
    primary_peak_amp = waveform[primary_peak_idx]
    wf_feat['peak_trough_ratio'] = primary_peak_amp / abs(primary_trough_amp)
    
    return wf_feat

In [8]:
# ==============================================================================
# 4. COMBINE ALL wf_feat
# ==============================================================================

all_wf_feat = []

for i, unit in enumerate(tqdm(unit_data, desc="Extracting all wf_feat")):
# for unit in [unit_data[41]]:
# for unit in unit_data[41:44]:
    wf_feat = {}
    
    # Spatial wf_feat
    spatial_feats = calculate_spatial_spread_improved(
        unit['multichan_waveforms'],
        unit['channel_depths'],
        unit['peak_depth'], 
        unit['peak_amp']
    )
    wf_feat.update(spatial_feats)
    
    # Temporal wf_feat from peak channel
    temporal_feats = extract_temporal_wf_feat(unit['peak_waveform'], sr, sbefore)
    wf_feat.update(temporal_feats)

    slope, r2 = calculate_conduction_velocity(unit['multichan_waveforms'], unit['channel_depths'], sr)
    wf_feat['conduction_slope'] = slope
    wf_feat['conduction_r2'] = r2
    
    # Add metadata
    wf_feat['unit_idx'] = i
    # wf_feat['cluster_id'] = good_clusters[i]
    wf_feat['peak_channel'] = unit['peak_ch']
    wf_feat['peak_depth'] = unit['peak_depth']
    
    all_wf_feat.append(wf_feat)

# Convert to DataFrame
wf_feat_df = pd.DataFrame(all_wf_feat)

print("\n=== FEATURE SUMMARY ===")
# Updated column names to match the new spatial wf_feat
print(wf_feat_df[['n_active_channels_10pct', 'n_active_channels_30pct',
                   'spatial_extent_10pct', 'spatial_extent_30pct', 
                   'decay_length', 'has_secondary_trough', 
                   'secondary_to_primary_ratio', 'trough_to_peak_time']].describe())


# Also print all available columns to confirm what we have
print("\n=== AVAILABLE wf_feat ===")
print(wf_feat_df.columns.tolist())


Extracting all wf_feat:   0%|          | 0/162 [00:00<?, ?it/s]


=== FEATURE SUMMARY ===
       n_active_channels_10pct  n_active_channels_30pct  spatial_extent_10pct  \
count               162.000000               162.000000            162.000000   
mean                 15.438272                 6.938272            245.648148   
std                  15.813329                 8.724700            302.264815   
min                   3.000000                 1.000000             30.000000   
25%                   9.000000                 4.000000            120.000000   
50%                  11.500000                 6.000000            157.500000   
75%                  16.000000                 8.000000            225.000000   
max                  96.000000                91.000000           1425.000000   

       spatial_extent_30pct  decay_length  has_secondary_trough  \
count            162.000000  1.620000e+02            162.000000   
mean              94.166667  7.953514e+05              0.203704   
std              154.617591  3.402656e+06   

In [9]:
# Simple classification

c1 = wf_feat_df['spatial_extent_10pct'] > 300
c2 = wf_feat_df['trough_to_peak_time'] < 0.24
c3 = wf_feat_df['n_fast_peaks'] > 1

sum_criteria = np.column_stack([c1, c2, c3]).sum(axis=1)

rgc_rows = sum_criteria > 1
sc_rows = sum_criteria < 2

celltype = np.where(rgc_rows, "RGC", "SC")

rgc_sum = rgc_rows.sum()
sc_sum = sc_rows.sum()
pct_rgc = rgc_sum/sc_sum * 100
pct_rgc = float('{:.3g}'.format(pct_rgc))

print(f'# RGC: {rgc_sum}, # SC: {sc_sum}, pct RGC: {pct_rgc}')


# RGC: 42, # SC: 120, pct RGC: 35.0


In [ ]:
# Save waveforms to pdf

plt.figure(figsize=(4, 6))

with PdfPages(f"R:\Basic_Sciences\Phys\SenzaiLab\Elissa_Belluccini\Matlab\MatFiles\SpikeWaveforms\PDF_Summary\{session}_waveforms.pdf") as pdf:
    for unit_idx in range(len(unitIds)):

        uid = unitIds[unit_idx]
        pk_ch = unit_data[unit_idx]['peak_ch']
        ch_idxs = unit_data[unit_idx]['ch_idxs']
        pk_idx = ch_idxs==pk_ch
        ch_ids = unit_data[unit_idx]['channels']
        ch_xp = xpos[ch_idxs]
        wf = unit_data[unit_idx]['multichan_waveforms']  # (r, 32)
        
        ct = celltype[unit_idx]
        peaks = (wf_feat_df['peaks'][unit_idx]+1) / sr*1000 # ms
        peaks_amp = wf_feat_df['peaks_amp'][unit_idx]
        ss = wf_feat_df['spatial_extent_10pct'][unit_idx]
        ttpt = wf_feat_df['trough_to_peak_time'][unit_idx]
        ttpt = float('{:.2g}'.format(ttpt))
        nfp = wf_feat_df['n_fast_peaks'][unit_idx]
        pti = (wf_feat_df['primary_trough_idx'][unit_idx]+1) / sr*1000
        pta = wf_feat_df['primary_trough_amp'][unit_idx]

        offset = np.max(np.abs(wf)) * 1.2
        u_ch_xp = np.unique(ch_xp)

        if len(u_ch_xp) > 1:
            x_shift = np.where(ch_xp == np.max(u_ch_xp), 4.2, 0)
        else:
            x_shift = np.zeros(len(ch_idxs))

        pk_i = np.where(pk_idx)[0][0]

        for i in range(wf.shape[0]):
            xv = timepts + x_shift[i]
            plt.plot(xv, wf[i] + i * offset, color='0.2', lw=1.2)

            if i == pk_i:
                plt.scatter(peaks + x_shift[i], peaks_amp + i * offset, color='c', s=8, zorder=3)
                plt.scatter(pti + x_shift[i], -pta + i * offset, color='b', s=8, zorder=3)

        yticks = np.arange(wf.shape[0]) * offset
        plt.yticks(yticks, [f'ch {ch_ids[i]}' for i in range(wf.shape[0])], fontsize=8)

        ax = plt.gca()
        ax.set_xticklabels([])
        ax.grid(True)
        xlim = ax.get_xlim()
        ylim = ax.get_ylim()

        # Plot x-axis scale bar 
        x_bar_len = 1  # ms
        x_bar_y = ylim[0] 
        x_bar_x0 = xlim[0] + 0.05 * (xlim[1] - xlim[0])

        plt.plot([x_bar_x0, x_bar_x0 + x_bar_len], [x_bar_y, x_bar_y], color='k', lw=1.5)
        plt.text(x_bar_x0, x_bar_y - 0.05*(ylim[1]-ylim[0]),
                '1 ms', fontsize=10)

        # Plot y-axis scale bar
        y_bar_len = 500  # µV
        y_bar_x = xlim[1] 
        y_bar_y0 = ylim[0] + 0.05 * (ylim[1] - ylim[0])

        plt.plot([y_bar_x, y_bar_x], [y_bar_y0, y_bar_y0 + y_bar_len], color='k', lw=1.5)
        plt.text(y_bar_x + 0.02*(xlim[1]-xlim[0]),
                y_bar_y0 + y_bar_len/2,
                '500 µV',
                rotation=90,
                va='center',
                fontsize=10)

        plt.title(f'uid: {uid}, {ct}, 10% spatial spread: {ss} µm, trough to peak time: {ttpt} ms, {nfp} peak(s)', fontsize=8) 
        # plt.show()

        pdf.savefig() # save current figure to PDF
        plt.close()

In [10]:
# Save data
wf_feat = wf_feat_df[
    ['unit_idx','peak_channel','peaks','peaks_amp',
     'spatial_extent_10pct','trough_to_peak_time','n_fast_peaks']
]

wf_feat['celltype'] = list(celltype)

wf_feat = wf_feat.to_dict('list')
wf_feat['celltype'] = np.array(wf_feat['celltype'], dtype=object)

sp = os.path.join(matlab_path,"wf_features", f"{session}.mat")
savemat(sp, {'wf_feat': wf_feat})  

C:\Users\blp0418\AppData\Local\Temp\ipykernel_36876\2809285303.py:7: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  wf_feat['celltype'] = list(celltype)
